# Checkpoint Week 1 — KNN &amp; Naive Bayes

Vul alle cellen in en push dit bestand naar je repo. De automatische checks (GitHub Actions) voeren dit notebook uit en controleren of alle **variabelen met de gevraagde namen** bestaan en correct zijn.

**BELANGRIJK**: gebruik exact de genoemde variabelennamen (accuracy_iris, mse_mall, spam_given_money, ...), anders faalt de controle.

---
# KNN

## Oefening 1 — Eigen KNN-implementatie

Implementeer KNNClassifier. De constructor krijgt k mee; fit bewaart de trainingsdata; predict geeft voor elk punt het meerderheidslabel van de k dichtste buren (Euclidische afstand).

In [ ]:
import numpy as np

class KNNClassifier:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        preds = []
        for x in X:
            dists = np.sqrt(((self.X_train - x) ** 2).sum(axis=1))
            idx = np.argsort(dists)[:self.k]
            labels, counts = np.unique(self.y_train[idx], return_counts=True)
            preds.append(labels[np.argmax(counts)])
        return np.array(preds)


### Extra oefening — KNN met euclidean_distance (scaffolding)

Gebruik deze versie om stap voor stap de Euclidische afstand te implementeren.

In [ ]:
import numpy as np

class KNNClassifierScaffold:
    def __init__(self, k=3):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        #todo
        self.X_train = X
        self.y_train = y

    def euclidean_distance(self, x1, x2):
        return np.sqrt(np.sum((x1 - x2)**2))

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):

        #bereken afstand van X tot alle trainingspunten
        dists = []
        for X_train in self.X_train:
            dists.append(self.euclidean_distance(x, X_train))

        dists = np.array(dists)

        #sorteer volgens klein naar groot
        order = np.argsort(dists)

        #de eerste K labels van 
        dichtstePunten = order[:self.k]
        labels = self.y_train[dichtstePunten]

        #gemiddelde terugsturen
        return np.mean(labels)

        #todo: 1 item voorspellen
        # bereken voor X de afstand met alle items van X_train
        # sorteren volgens van klein naar groot volgens die afstand
        # de eerste k labels (y data): neem daarvan het gemiddelde
        # gebruik np.argsort, np.mean


# Sample dataset - house prices
X_train = np.array([
    [0, 0, 1000, 2, 1],
    [1, 1, 1500, 3, 2],
    [2, 2, 1200, 2, 1],
    [3, 3, 1800, 4, 2],
    [4, 4, 2000, 3, 2]
])
y_train = np.array([200000, 250000, 220000, 280000, 300000])
X_test = np.array([
    [5, 5, 1600, 3, 1],
    [2, 3, 1300, 2, 1]
])

knn_scaffold = KNNClassifierScaffold(k=3)
knn_scaffold.fit(X_train[:, 1:], y_train)
predictions = knn_scaffold.predict(X_test[:, 1:])
print("Predicted house prices:", predictions)

## Oefening 2 — Iris met sklearn-KNN

- Laad de iris-dataset, splits in train/test (test_size=0.25, random_state=42).
- **Normaliseer** met een StandardScaler (fit op train alleen!).
- Train een KNeighborsClassifier(n_neighbors=5).
- Sla de voorspellingen op in y_pred_iris en de **accuracy op de testset** in accuracy_iris (float).

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

#laad dataset iris van import
X, y = load_iris(return_X_y=True)

#split data op met test_size 25% en seed 42
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

#normaliseer de trainingsdata
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

#fit de KNN met de dataset en labels
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_scaled, y_train)

#predict de scaled test data via de knn method
y_pred_iris = knn.predict(X_test_scaled)

#projecteer een accuracy score
accuracy_iris = accuracy_score(y_test, y_pred_iris)
print(accuracy_iris)



**Vraag (antwoord in antwoord_normaliseren)**: Waarom moet je de features normaliseren vóór KNN? Kies A, B, C of D:
- A: Omdat KNN alleen met integers werkt
- B: Omdat afstanden anders vertekend worden door schaalverschillen tussen features
- C: Omdat accuracy dan altijd 100% wordt
- D: Omdat sklearn dat vereist voor alle modellen

In [ ]:
antwoord_normaliseren = "normaliseren voorkomt vertekening van afstanden door schaalverschillen"

### Iris — stap voor stap (extra scaffolding)

Werk deze cellen uit om het effect van de StandardScaler te onderzoeken.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
# Dataset inladen
iris = load_iris()
X = iris.data
y = iris.target
print(iris.keys())

# split data op in train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=40)

#scale X_train data naar universele waardes
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

#zet het model op met juist aantal neighbours en fit het met de training data EN LABELS!!!
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train_scaled,y_train)

# voorspel de soort planten van onze test data
y_predicted = neigh.predict(X_test_scaled)
#print(y_predicted)

#kijk na hoe correct ons model was door middel van de test labels te vergelijken met de voorspellingen.
accuracy = accuracy_score(y_test, y_predicted)

print(accuracy)

print(classification_report(y_test, y_predicted))

In [ ]:
# zoek op hoe je data kan train-test splitten
# We importen vanuit: sklearn.model_selection de train_test_split methode
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=40)

In [ ]:
# Gebruik een standardscaler op de X_train, X_test data
# We importen vanuit: sklearn.preprocessing de standardScaler methode
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Gebruik een KNeighborsclassifier van de sklearn library. Fit deze. Maak voorspellingen.
# We importen vanuit: sklearn.neighbours de KNeighborsClassifier methode
neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X_train_scaled,y_train)

y_predicted = neigh.predict([[0.2, 0.2, 0.8, 0.7]])

In [ ]:
# Bereken de accuracy.
# We importen vanuit: sklearn.metrics de accuracy_score methode

y_predicted = neigh.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_predicted)

print(accuracy)

In [ ]:
# maak een classificatie report
# We importen vanuit: sklearn.metrics de classification_report methode
print(classification_report(y_test, y_predicted))

## Oefening 3 — Mall Customers (one-hot + regressie)

- Laad data/Mall_Customers.csv (staat in dezelfde map als dit notebook).
- One-hot encode Gender, voeg samen met Age en Annual Income (k$).
- Doelvariabele: Spending Score (1-100) : gebruik een KNeighborsRegressor(n_neighbors=5) met train/test split (test_size=0.25, random_state=42).
- Sla de test-voorspellingen op in y_pred_mall en de **MSE** in mse_mall (float).

In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

#laad dataset in
df = pd.read_csv("data/Mall_Customers.csv")
print(df)

#one-hot encode Gender, voeg dit samen met Age & Annual income
# NOTE!!!! FIT_TRANSFORM ACTUALLY CHANGES THE VALUES!! FIT BY ITSELF DOES NOT!
enc = OneHotEncoder(drop='first', sparse_output=False)
df_categorical = df[['Gender']]
gender_enc = enc.fit_transform(df_categorical)

# NOTE ORRRR
# gender_enc = enc.fit_transform(df[['Gender']])
# THIS ALSO WORKS AND IS FASTER THAN DEFINING AN EXTRA VARIABLE

X_mall = pd.DataFrame(gender_enc, columns=['Gender_Encoded'])
X_mall[['Age', 'Annual Income (k$)']] = df[['Age', 'Annual Income (k$)']].values
y = df['Spending Score (1-100)']

print()
print(X_mall)

#Doelvariabele: Spending Score (1-100): gebruik een KNeighborsRegressor(n_neighbors=5) met train/test split (test_size=0.25, random_state=42).
neigh = KNeighborsRegressor(n_neighbors=5)

X_train, X_test, y_train, y_test = train_test_split(X_mall,y,test_size=0.25,random_state=42)

neigh.fit(X_train,y_train)

#sla test op in y_pred mall en MEAN SQUARE ERROR in mse_mall
y_pred_mall = neigh.predict(X_test)
mse_mall = mean_squared_error(y_test, y_pred_mall)

print(mse_mall)





     CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0             1    Male   19                  15                      39
1             2    Male   21                  15                      81
2             3  Female   20                  16                       6
3             4  Female   23                  16                      77
4             5  Female   31                  17                      40
..          ...     ...  ...                 ...                     ...
195         196  Female   35                 120                      79
196         197  Female   45                 126                      28
197         198    Male   32                 126                      74
198         199    Male   32                 137                      18
199         200    Male   30                 137                      83

[200 rows x 5 columns]

     Gender_Encoded  Age  Annual Income (k$)
0               1.0   19                  15
1        

**Vraag (antwoord in antwoord_k_kiezen)**: Hoe kies je K het best? Kies A, B, C of D:
- A: Altijd K=1
- B: Zomaar op het gevoel
- C: Via cross-validatie
- D: K moet altijd gelijk zijn aan het aantal klassen

In [ ]:
antwoord_k_kiezen = "C"

### Mall customers — stap voor stap (extra scaffolding)

Bekijk de data van de Mall_customers.csv - laad deze in en bekijk de kolommen. Pas hier KNN toe (laatste kolom spending score is het te voorspellen label). Wat moet je nog aanpassen om dit goed te laten werken?

In [ ]:
#code
import pandas as pd

df = pd.read_csv('data/Mall_Customers.csv')


Meer informatie over One-Hot encoding: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html

In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd
data = pd.read_csv('data/Mall_Customers.csv')

In [ ]:
data.head()
X_numerical = data[['Age', 'Annual Income (k$)']]
X_categorical = data[['Gender']]
y = data[['Spending Score (1-100)']]

Gebruik one-hot encoding om de categorische variabelen om te zetten naar numerieke variabelen.

In [ ]:
#todo
# gender_enc = enc.fit_transform(df[['Gender']])
gender_enc = enc.fit_transform(X_categorical)

In [10]:
# terug samenvoegen met numerieke variabelen
X_mall = pd.DataFrame(gender_enc, columns=['Gender_Encoded'])
X_mall[['Age', 'Annual Income (k$)']] = df[['Age', 'Annual Income (k$)']].values

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_mall,y,test_size=0.25,random_state=42)

# Maak KNN regressor
neigh = KNeighborsRegressor(n_neighbors=5)

# Train model
neigh.fit(X_train,y_train)

# Voorspelling
y_pred_mall = neigh.predict(X_test)

# Evaluatie met mean squared error en r2 score
mse_mall = mean_squared_error(y_test, y_pred_mall)
r2_score_values = r2_score(y_test, y_pred_mall)
print(mse_mall)
print(r2_score_values)




393.62
0.3316734684452486


---
# Naive Bayes

## Oefening 4 — Bayes-formule

We beginnen met een simpele bayes functie die gegeven een aantal conditionele kansen en de algemene kans van een binair label, de kans berekent op een label gegeven de geziene data. Dit is puur de formule van Bayes invullen.

In [14]:
def bayes(conditional_prob_label, conditional_other_label_prob, prob_of_event):
    #TODO: een hoop factoren met elkaar vermenigvuldigen
    numerator = conditional_prob_label * prob_of_event
    denominator = numerator + (conditional_other_label_prob * (1 - prob_of_event))
    return (numerator / denominator)

spam_given_money = bayes(0.4, 0.01, 0.45)
print(spam_given_money)

0.9703504043126684


Voor de classificatie moet je de noemer zelfs niet uitrekenen. De classifier zoekt de grootste teller en geeft dat terug als antwoord. We kijken nu naar een classificatie met meerdere woorden (realistischer).

In [22]:
#             IN SPAM | IN HAM
given_money = [0.4, 0.00001]
uganda = [0.1, 0.05]
ap_hogeschool = [0.001, 0.3]
conditional_data = [given_money, uganda, ap_hogeschool]
prob_spam = 0.45
data_vector = [1, 1, 0] # given_money en uganda, geen 'ap_hogeschool'

def bayes_classifier_spam(conditional_data, prob_spam, data_vector):
    #checken voor SPAM:
    spam_probability = prob_spam
    for i in range(len(conditional_data)):
        if data_vector[i] == 1:
            spam_probability *= conditional_data[i][0]
        else:
            spam_probability *= (1 - conditional_data[i][0])
    print(f"spam prob: {spam_probability}")
    #checken voor HAM:
    not_spam_probability = 1-prob_spam
    for i in range(len(conditional_data)):
        if data_vector[i] == 1:
            not_spam_probability *= conditional_data[i][1]
        else:
            not_spam_probability *= (1 - conditional_data[i][1])
    print(f"NOTspam prob: {not_spam_probability}")
    if (spam_probability > not_spam_probability):
        return "SPAM"
    else:
        return "HAM"

is_spam = bayes_classifier_spam(conditional_data, prob_spam, data_vector)
print("Is Spam?:", is_spam)

spam prob: 0.017982
NOTspam prob: 1.9250000000000004e-07
Is Spam?: SPAM


## Oefening 5 — Naive Bayes met sklearn (BernoulliNB)

We gaan deze code nu laten werken door middel van de ingebouwde sci-kit learn implementatie.

In [ ]:
import numpy as np
from sklearn.naive_bayes import BernoulliNB

# Given data
given_money = [0.4, 0.00001]
uganda = [0.1, 0.05]
ap_hogeschool = [0.001, 0.3]
conditional_data = [given_money, uganda, ap_hogeschool]
prob_spam = 0.45
data_vector = [0, 1, 1]  # uganda en AP Hogeschool, geen 'moneytransfer'

In [ ]:
import numpy as np
from sklearn.naive_bayes import BernoulliNB
# Data naar het juiste formaat brengen
X = np.array([
    [1, 0, 0],  # voorbeeld 'given_money'
    [0, 1, 0],  # 'uganda'
    [0, 0, 1],  # 'ap_hogeschool'
])

# bepaal de bijbehorende kansen met np.array:
y_spam = np.array([conditional_data[i][0] for i in range(len(conditional_data))])
y_ham = np.array([conditional_data[i][1] for i in range(len(conditional_data))])

# kleef de trainingsdata op elkaar met np.vstack en np.hstack
X_train = np.vstack([X, X])
y_train = np.hstack([np.ones(len(y_spam)), np.zeros(len(y_ham))])

print(X_train)
print()
print(y_train)

[[1 0 0]
 [0 1 0]
 [0 0 1]
 [1 0 0]
 [0 1 0]
 [0 0 1]]

[1. 1. 1. 0. 0. 0.]


In [34]:
# train hier een Naive bayes classifier, en gebruik hem om voor de data_vector te voorspellen of het ham (0) of spam (1) is
# TODO: nb_clf = BernoulliNB().fit(X_train, y_train)
# TODO: y_pred = nb_clf.predict(...)
nb_clf = BernoulliNB().fit(X_train,y_train)
y_pred = nb_clf.predict(X_train)

print(y_pred)

[0. 0. 0. 0. 0. 0.]


## Oefening 6 — NaiveBayesianClassifier

We maken nu een volledige classifier aan, die we ook kunnen hertrainen.

In [ ]:
class NaiveBayesianClassifier:
    def __init__(self):
        self.labels = ['spam', 'ham', 'ongekend']
        self.label_probabilities = {label: 1/3 for label in self.labels}
        self.feature_probabilities = {label: {} for label in self.labels}
        # print(f"FEATURE START{self.feature_probabilities}")
    
    def train(self, label, feature_probabilities):
        #todo: trainen is gelabelde data toevoegen en kansen updaten
        for key, value in feature_probabilities.items():
            self.feature_probabilities[label][key] = value
    
    def classify(self, features):
        label_scores = {}

        for label in self.labels:
            probability = self.label_probabilities[label]

            for feature, value in features.items():
                feature_probability = self.feature_probabilities[label][feature][value]
                probability *= feature_probability

            label_scores[label] = probability
        print(label_scores.values())
        return max(label_scores, key=label_scores.get)

# Example usage
classifier = NaiveBayesianClassifier()

# Training data
training_data = {
    'spam': {
        'hair-extensions': [0.9, 0.2, 0.8],
        'dammen': [0.8, 0.1, 0.3]
    },

    'ham': {
        'hair-extensions': [0.1, 0.8, 0.2],
        'dammen': [0.2, 0.9, 0.7]
    },

    'ongekend': {
        'hair-extensions': [0.5, 0.4, 0.6],
        'dammen': [0.4, 0.5, 0.4]
    }
}

for label, data in training_data.items():
    for feature, probabilities in data.items():
        # classifier.train(label, {feature: x for x in probabilities}) 
        classifier.train(label, {feature: probabilities}) 

# Classificatie
input_features = {'hair-extensions': 0, 'dammen': 1}
result = classifier.classify(input_features)
print("Geklassificeerd als:", result)


dict_values([0.24, 0.006666666666666667, 0.06666666666666667])
Geklassificeerd als: spam
